In [68]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import csv
from tqdm import tqdm

In [69]:
# # ==========================
# # TEXT CLEANING
# # ==========================

# def clean_text(text):
#     if not text:
#         return ""

#     text = text.replace(",", "")
#     text = text.replace('"', "")
#     text = text.replace("\n", " ")
#     text = re.sub(r"\s+", " ", text)

#     return text.strip()


# # ==========================
# # NEWSEVENTS SCRAPER
# # ==========================

# def scrape_newsevents(soup):

#     title = ""
#     date = ""
#     speaker = ""
#     content = ""

#     # Title
#     title_tag = soup.find("h3", class_="title")
#     if title_tag:
#         title = clean_text(title_tag.get_text())

#     # Date
#     date_tag = soup.find("p", class_="article__time")
#     if date_tag:
#         date = clean_text(date_tag.get_text())

#     # Speaker
#     speaker_tag = soup.find("p", class_="speaker")
#     if speaker_tag:
#         speaker = clean_text(speaker_tag.get_text())

#     # Content
#     content_div = soup.find("div", class_="col-xs-12 col-sm-8 col-md-8")
#     if content_div:
#         paragraphs = content_div.find_all("p")
#         content = clean_text(
#             " ".join(
#                 p.get_text()
#                 for p in paragraphs
#                 if p.get_text().strip() != ""
#             )
#         )

#     return title, date, speaker, content


# # ==========================
# # BOARDDOCS SCRAPER
# # ==========================

# def scrape_boarddocs(soup):

#     title = ""
#     speaker = ""
#     date = ""
#     content = ""

#     tables = soup.find_all("table", width="600")

#     if not tables:
#         return "", "", "", ""

#     # ======================================
#     # SPEAKER + DATE (FIRST TABLE)
#     # ======================================

#     header_text = tables[0].get_text(" ", strip=True)

#     # Speaker
#     header_text = tables[0].get_text("\n", strip=True)

#     lines = header_text.split("\n")

#     for line in lines:
#         if "Remarks by" in line:
#             speaker_line = line.replace("Remarks by", "").strip()
#             speaker = clean_text(speaker_line)
#             break

#     # Date
#     date_match = re.search(r"\w+ \d{1,2}, \d{4}", header_text)
#     if date_match:
#         date = clean_text(date_match.group(0))

#     # ======================================
#     # TITLE (LOOK ANYWHERE SAFELY)
#     # ======================================

#     # First try: <font size="+1">
#     title_tag = soup.find("font", attrs={"size": "+1"})
#     if title_tag:
#         title = clean_text(title_tag.get_text())

#     # Fallback: <i>
#     if not title:
#         title_i = soup.find("i")
#         if title_i:
#             title = clean_text(title_i.get_text())

#     # Fallback: <title> tag parsing
#     if not title:
#         if soup.title:
#             raw_title = soup.title.get_text()
#             # Extract part after --
#             match = re.search(r"-- (.*?) --", raw_title)
#             if match:
#                 title = clean_text(match.group(1))

#     # ======================================
#     # CONTENT (ALL PARAGRAPHS INSIDE TABLES)
#     # ======================================

#     content_parts = []

#     for tbl in tables:
#         paragraphs = tbl.find_all("p")

#         for p in paragraphs:
#             text = p.get_text().strip()

#             if (
#                 text
#                 and "Return to top" not in text
#                 and "Speeches" not in text
#                 and "Last update" not in text
#                 and "Home |" not in text
#             ):
#                 content_parts.append(text)

#     content = clean_text(" ".join(content_parts))

#     return title, date, speaker, content

# # ==========================
# # MAIN PIPELINE
# # ==========================

# df = pd.read_csv("dataset/fed_speech.csv")

# results = []

# for link in tqdm(df["link"]):

#     try:
#         response = requests.get(link, timeout=15)
#         soup = BeautifulSoup(response.text, "html.parser")

#         if "newsevents" in link:
#             title, date, speaker, content = scrape_newsevents(soup)

#         elif "boarddocs" in link:
#             title, date, speaker, content = scrape_boarddocs(soup)

#         else:
#             title, date, speaker, content = "", "", "", ""

#         results.append({
#             "link": link,
#             "title": title,
#             "date": date,
#             "speaker": speaker,
#             "content": content
#         })

#     except Exception as e:
#         print(f"Error scraping {link}: {e}")
#         results.append({
#             "link": link,
#             "title": "",
#             "date": "",
#             "speaker": "",
#             "content": ""
#         })


# output_df = pd.DataFrame(results)

# # Let pandas handle quoting properly
# output_df.to_csv(
#     "dataset/fed_speeches_scraped.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL
# )

# print("Scraping complete.")


In [ ]:
# scrapped = pd.read_csv("dataset/fed_speech_content.csv")

In [ ]:
# scrapped.isna().sum()

link        0
title      62
date        6
speaker    12
content    14
dtype: int64

In [ ]:
# missing_mask = scrapped[
#     ["title", "date", "speaker", "content"]
# ].isna().any(axis=1)

# missing_rows = scrapped[missing_mask].copy()
# print(len(missing_rows))

79


In [ ]:
# # Fix missing links

# def scrape_boarddocs(soup):

#     def clean(text):
#         if not text:
#             return ""
#         text = text.replace(",", "")
#         text = text.replace('"', "")
#         text = text.replace("\n", " ")
#         text = re.sub(r"\s+", " ", text)
#         return text.strip()

#     title = ""
#     speaker = ""
#     date = ""
#     content = ""

#     full_text = soup.get_text("\n")

#     # ======================================
#     # 1️⃣ TITLE
#     # ======================================

#     # Try font +1
#     font_title = soup.find("font", attrs={"size": "+1"})
#     if font_title:
#         title = clean(font_title.get_text())

#     # Try parsing from <title>
#     if not title and soup.title:
#         raw = soup.title.get_text()
#         # Pattern: FRB: Speech, X -- TITLE -- DATE
#         match = re.search(r"--\s*(.*?)\s*--", raw)
#         if match:
#             title = clean(match.group(1))

#     # ======================================
#     # 2️⃣ SPEAKER
#     # ======================================

#     speaker_patterns = [
#         r"Remarks by (.+)",
#         r"Statement by (.+)",
#         r"Testimony of (.+)",
#         r"Address by (.+)"
#     ]

#     for pattern in speaker_patterns:
#         match = re.search(pattern, full_text)
#         if match:
#             speaker = clean(match.group(1).split("\n")[0])
#             break

#     # Fallback: first font size +2
#     if not speaker:
#         font_speaker = soup.find("font", attrs={"size": "+2"})
#         if font_speaker:
#             speaker = clean(font_speaker.get_text())

#     # ======================================
#     # 3️⃣ DATE
#     # ======================================

#     date_match = re.search(
#         r"\b\w+\s+\d{1,2},?\s+\d{4}\b",
#         full_text
#     )
#     if date_match:
#         date = clean(date_match.group(0))

#         # ======================================
#     # 4️⃣ CONTENT (FINAL HYBRID VERSION)
#     # ======================================

#     content = ""
#     tables = soup.find_all("table", width="600")

#     # --------------------------------------
#     # METHOD 1: Paragraph extraction (works for many)
#     # --------------------------------------
#     content_parts = []

#     for tbl in tables:
#         paragraphs = tbl.find_all("p")

#         for p in paragraphs:
#             text = p.get_text(strip=True)

#             if not text:
#                 continue

#             if any(skip in text for skip in [
#                 "Return to top",
#                 "Speeches",
#                 "Last update",
#                 "Accessibility",
#                 "Contact Us",
#                 "Home |"
#             ]):
#                 continue

#             content_parts.append(text)

#     content = clean(" ".join(content_parts))

#     # --------------------------------------
#     # METHOD 2: If paragraph method too small → use full table text
#     # --------------------------------------
#     if len(content) < 500 and len(tables) >= 2:

#         main_table = tables[1]
#         raw_text = main_table.get_text(" ", strip=True)

#         stop_phrases = [
#             "Return to top",
#             "Speeches",
#             "Accessibility",
#             "Last update",
#             "Home |"
#         ]

#         for phrase in stop_phrases:
#             if phrase in raw_text:
#                 raw_text = raw_text.split(phrase)[0]

#         alt_content = clean(raw_text)

#         # Use whichever is longer (safer)
#         if len(alt_content) > len(content):
#             content = alt_content

#     # --------------------------------------
#     # METHOD 3: Last fallback – global paragraphs
#     # --------------------------------------
#     if not content:

#         fallback_parts = []

#         for p in soup.find_all("p"):
#             text = p.get_text(strip=True)

#             if not text:
#                 continue

#             if any(skip in text for skip in [
#                 "Return to top",
#                 "Speeches",
#                 "Last update",
#                 "Accessibility",
#                 "Contact Us",
#                 "Home |"
#             ]):
#                 continue

#             if len(text) < 40:
#                 continue

#             fallback_parts.append(text)

#         content = clean(" ".join(fallback_parts))

#     return (
#         title,
#         date,
#         speaker,
#         content
#     )


In [ ]:
# for idx in missing_rows.index:
#     link = scrapped.loc[idx, "link"]

#     response = requests.get(link, timeout=10)
#     soup = BeautifulSoup(response.text, "html.parser")

#     title, date, speaker, content = scrape_boarddocs(soup)

#     scrapped.loc[idx, ["title","date","speaker","content"]] = \
#         [title, date, speaker, content]

In [ ]:
# scrapped.to_csv(
#     "try.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL
# )

In [ ]:
# import pandas as pd

# scrapped = pd.read_csv("try.csv")

# # Create mask for weak content
# mask = scrapped["content"].str.len() < 500

# # See how many will be cleared
# print("Rows to clear:", mask.sum())

# # Set them to empty string
# scrapped.loc[mask, "content"] = ""

# # (Optional) Verify
# print((scrapped["content"].str.len() < 500).sum())

# scrapped.to_csv(
#     "try.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL
# )

Rows to clear: 10
10


Rest is all manually filled

In [47]:
import pandas as pd

scrapped = pd.read_csv("dataset/fed_speech_content.csv")

mask1 = (
    scrapped["content"].fillna("").str.len() < 500
) 

mask2 = (
    scrapped["date"].fillna("").str.len() > 17
)

mask3 = (
    scrapped["speaker"].fillna("").str.len() > 100
)

mask4 = (
    scrapped["title"].fillna("").str.len() > 250
)

print("Number of matching rows:", len(scrapped[mask1]))
print("Number of matching rows:", len(scrapped[mask2]))
print("Number of matching rows:", len(scrapped[mask3]))
print("Number of matching rows:", len(scrapped[mask4]))

Number of matching rows: 0
Number of matching rows: 0
Number of matching rows: 0
Number of matching rows: 0


In [ ]:
scrapped['date'] = pd.to_datetime(scrapped['date'], format='%Y-%m-%d')
scrapped.sort_values(by="date")